In [1]:
from f_0_dirs import get_data_dirs
dirs = get_data_dirs()

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


# 1. [read] from FAME

### Categorise raw files

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [2]:
from flask import json
import pandas as pd

def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60
    return "%dh:%02dm:%02ds" % (hour, minutes, seconds)
rate = 1 # 1 second per file

from f_1_traverse import build_raw_file_dict
pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict, processed = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")
    print(f"✅ Processed {processed} file paths.")
    print(f"⏱️ Estimated time to process all files: {convert(processed * rate)} (at {rate} second per file)")

Traversing industry directory: 01, 02, 03, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16
17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31
32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 45, 46, 47, 49, 50
51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68
69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 84, 85
86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
✅ Successfully built raw file dictionary and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\raw_file_dict.json
✅ Processed 12846 file paths.
⏱️ Estimated time to process all files: 3h:34m:06s (at 1 second per file)


### Process each excel file in the raw file dictionary

In [3]:
from typing import TypedDict
import pandas as pd

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_source df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.root_dir / "build" / "input" / "raw_properties.xlsx"
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="calamine")

# Schema for raw inputs is rows where from_raw is not blank
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]

# Take our master schema and turn it into a helpful mapping of fuzzy column names to schema column names
# Regardless of what the raw source is. We'll filter it later
class SRFbyProperty(TypedDict):
    mapping: pd.DataFrame
    col_map: dict[str, str]

properties = ['a1_ID', 'a2_key_finance', 'a3_assets', 'a4_profits', 'a5_misc']
schema_fixed_fuzzy_df: pd.DataFrame = schema_source[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_df["fml"] = schema_fixed_fuzzy_df["fuzzy_mapping"].str.split('\n')
schema_fixed_fuzzy_by_property = {}
for p in properties:
    srf_filtered: pd.DataFrame = schema_raw[schema_raw["from_raw"].isin([p, 'all'])]
    srf_filtered["fml"] = srf_filtered["fuzzy_mapping"].str.split('\n')
    # Just make 
    srf_mapping: pd.DataFrame = srf_filtered
    # For every value in srf_filtered["fml"]
    # Create a dict which is that value, and the corresponding entry in srf_filtered["key"]
    srf_col_map = {
        fml: key
        for _, row in srf_filtered.iterrows()
        for fml in row["fml"]
        for key in [row["key"]]
    }
    schema_fixed_fuzzy_by_property[p] = {
        "mapping": srf_mapping,
        "col_map": srf_col_map
    }

# Print any rows where fml has more than one element
for index, row in schema_fixed_fuzzy_df.iterrows():
    if row["fml"] is None:
        print(f"Row {index} has fml that is None: {row['key']}")
    elif type(row["fml"]) is not list:
        print(f"Row {index} has fml that is not a list: {row['key']}")
    elif len(row["fml"]) > 1:
        print(f"Row {index} has more than one fuzzy mapping: {row['fml']}")

Row 37 has more than one fuzzy mapping: ['Strategy,  organization and policy', 'Strategy, organization and policy']
Row 49 has fml that is not a list: has_ptaddress
Row 50 has fml that is not a list: has_ptaddress_latlong
Row 51 has fml that is not a list: is_public
Row 52 has fml that is not a list: has_company_branch_mismatch
Row 53 has fml that is not a list: industry_code
Row 54 has fml that is not a list: file_code
Row 55 has fml that is not a list: fame_key


# 2. [write] To duck schemas

### Define duck schemas

In [4]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb,geospatial]'

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# Fixed schema
# schema_fixed is a df of schema_source where values in column "keep" are "fixed" or "all"
schema_fixed: pd.DataFrame = schema_source[schema_source["keep"].isin(["fixed", "all"])]
schema_fixed_dict: dict[str, str] = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis: ibis.Schema = ibis.schema(schema_fixed_dict)
schema_fixed_names: set[str] = set(schema_fixed_ibis.keys())

schema_derived: pd.DataFrame = schema_source[schema_source["keep"].isin(["derived", "all"])]
schema_derived_dict: dict[str, str] = dict(zip(schema_derived["key"], schema_derived["type"]))
schema_derived_ibis: ibis.Schema = ibis.schema(schema_derived_dict)
schema_derived_names: set[str] = set(schema_derived_ibis.keys())

# build a schema_yearly df with columns registered_number, fame_key, year, value properties, with types str, str, int, float
schema_yearly: pd.DataFrame = pd.DataFrame({
    "key": ["registered_number", "fame_key", "year", "value"],
    "type": ["string", "string", "int64", "float64"]
})
schema_yearly_dict: dict[str, str] = dict(zip(schema_yearly["key"], schema_yearly["type"]))
schema_yearly_ibis: ibis.Schema = ibis.schema(schema_yearly_dict)
schema_yearly_names: set[str] = set(schema_source[
    (schema_source["keep"] == "yearly") &
    (schema_source["from_raw"].notna())]["key"]
)

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_fixed", schema=schema_fixed_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_fixed'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_fixed").schema())

    con.create_table("fame_derived", schema=schema_derived_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_derived'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_derived").schema())

    con.create_table("fame_yearly", schema=schema_yearly_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_yearly'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_yearly").schema())

except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
✅ Successfully created Ibis schema for 'fame_fixed'.

Table Schema Verification:
ibis.Schema {
  company_name                       string
  registered_number                  string
  ticker_symbol                      string
  primary_trading_address            string
  primary_trading_address_latitude   string
  primary_trading_address_longitude  string
  branch_name                        string
  primary_uk_sic_2007_code           int64
  primary_uk_sic_2007_description    string
  latest_accounts_date               date
  no_of_available_years              int64
  guo                                string
  guo_nb                             int64
  entity_type                        string
}
✅ Successfully created Ibis schema for 'fame_derived'.

Table Schema Verifica

### Load, modify and write imported file schema

In [ ]:
import traceback

from flask import json
from f_1_traverse import RawFileDict
from f_2_check import drop_duplicate_columns, check_df_matches_schema, handle_excel_dates, rename_df_with_years
from f_2_modify import coerce_ibis_dates_from_schema, reindex_ibis_table
import random

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

# Handle yearly variables
start_year = 2006
end_year = 2025

# We want one big dataframe, which we will merge all the data into for now
# df_fixed = pd.DataFrame(columns=list(schema_fixed.columns))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

process_count = 0
for ind, obj in raw_file_dict.items():
    print(f"Ingesting industry: {ind} with {len(obj)} properties.")
    process_count += 1
    if process_count > 3:
        break

    batches_raw: dict[str, list[pd.DataFrame]] = { p: [] for p in properties }
    for property, arr in obj.items():
        print(f"--- Ingesting property: {property} with {len(arr)} files.")
        schema_raw_fuzzy_col_map = schema_fixed_fuzzy_by_property[property]['col_map']

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for [file_name, file_path] in files_shuffled:

            # LOAD: df_raw has no fixed schema so we can ingest and modify it as pleases
            df_raw = pd.read_excel(file_path, engine='calamine', sheet_name='Results', header=0, dtype={
                "registered_number": str                                            # "Leading Zeros" Trap. Pandas accidentally processes
            })                                                                      #   registered number as int64 when reading the file, which will chop zeros.
            df_raw.drop(df_raw.columns[0], axis=1, inplace=True)                    # Drop column A (blank in raw data)
            df_raw = drop_duplicate_columns(df_raw)                                 # Remove duplicate columns (quirk of some files)

            df_raw = rename_df_with_years(
                df_raw, schema_raw_fuzzy_col_map, property, start_year, end_year,
                ref=f"{ind}/{property}/{file_name}"
            )
            # # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # # Doesn't match the current property or 'all'. This ensures we only keep relevant columns for the current property.
            # p_cols = schema_raw[schema_raw["from_raw"].isin([property, 'all'])]["key"].tolist()
            # df_raw = df_raw[p_cols]

            # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # Doesn't match the current property or 'all'.
            # Unless the 'keep' column is set to yearly
            # In which case the column name only has to start with the 'key' for that property
            # This ensures we only keep relevant columns for the current property.
            exact_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] != "yearly")]["key"].tolist()
            yearly_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] == "yearly")]["key"].tolist()
            df_raw_cols = [col for col in df_raw.columns if col in exact_cols or any(col.startswith(yc) for yc in yearly_cols)]
            df_raw = df_raw[df_raw_cols]

            df_raw = df_raw[df_raw['registered_number'].notna()]      
            df_raw = handle_excel_dates(df_raw)                                     # Handle any Excel date serials to datetime
            
            df_raw['industry_code'] = ind
            df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
            batches_raw[property].append(df_raw)

    print(f"Deriving ingested files for industry: {ind}.")

    # START DERIVING
    ind_batches_masters = {
        p: pd.concat(
            batches_raw[p], ignore_index=True
        ).drop_duplicates(                              # Drop duplicates based on 'registered_number'
            subset=['registered_number'], keep='first'
        ).copy()
        for p in properties if batches_raw.get(p)
    }
    try:
        for p in properties:
            if p in ind_batches_masters:
                continue
            raise ValueError(f"❌ Error: No data found for property '{p}' in industry '{ind}'. Please check the raw data files.")

        # FILTER, processing the a1_ID table first.
        schema_raw_fuzzy_mapping = schema_fixed_fuzzy_by_property["a1_ID"]["mapping"]
        df_id = ind_batches_masters["a1_ID"]
        if 'no_of_available_years' in df_id.columns:
            df_id = df_id[df_id['no_of_available_years'] != 0]
        if 'ro_country' in df_id.columns:
            df_id = df_id[df_id['ro_country'] != "Republic of Ireland"]
        ind_batches_masters["a1_ID"] = df_id

        try:
            check_df_matches_schema(schema_raw_fuzzy_mapping, ind_batches_masters["a1_ID"])           # Check if the DataFrame matches the input
        except ValueError as e:
            print(f"⚠️ Mismatch in raw data validation for file {ind}/a1_ID)")
            print("Warning:", e)

        # MODIFY
        table_t_id = ibis.memtable(ind_batches_masters["a1_ID"])
        table_t_misc = ibis.memtable(ind_batches_masters["a5_misc"])
        table_t_merged_id_misc = table_t_id.left_join(table_t_misc, "registered_number").select(
            *[table_t_id[col] for col in table_t_id.columns],
            *[table_t_misc[col] for col in table_t_misc.columns if col not in table_t_id.columns]
        )
        table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_merged_id_misc)
        table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        table_fixed_type_casts = {
            col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
            for col in schema_fixed_names if col in table_t_fixed_dates.columns
        }
        table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        con.insert("fame_fixed", table_t_fixed_cast)
        print(f"✅ Successfully appended fixed table from: {ind}")

        # for p in ["a1_ID", "a5_misc"]: 
        #     df_master = ind_batches_masters[p]
        #     table_t_raw: ibis.expr.types.Table = ibis.memtable(df_master)          # Could be any property, set the raw table into memory

        #     # Filter the memtable to only columns that exist in the target schema
        #     table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_raw)
        #     table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        #     table_fixed_type_casts = {
        #         col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
        #         for col in schema_fixed_names if col in table_t_fixed_dates.columns
        #     }
        #     table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        #     con.insert("fame_fixed", table_t_fixed_cast)
        #     print(f"✅ Successfully appended fixed table from: {ind}")

        # DERIVE
        # for p in ["a1_ID"]:
            # table_derived = table_t_raw.mutate(
            #     has_ptaddress = table_t_raw.primary_trading_address.notnull(),
            #     has_ptaddress_latlong = table_t_raw.primary_trading_address_latitude.notnull()
            #         & table_t_raw.primary_trading_address_longitude.notnull(),
            #     is_public = table_t_raw.ticker_symbol.notnull(),
            #     has_company_branch_mismatch = table_t_raw.company_name != table_t_raw.branch_name
            # ).select(schema_derived_names) # Assuming schema_derived_names is a list of your derived cols
            
            # Safely extract columns, defaulting to an Ibis null object if missing from raw data
        col_pta = table_t_id['primary_trading_address'] if 'primary_trading_address' in table_t_id.columns else ibis.null()
        col_lat = table_t_id['primary_trading_address_latitude'] if 'primary_trading_address_latitude' in table_t_id.columns else ibis.null()
        col_lon = table_t_id['primary_trading_address_longitude'] if 'primary_trading_address_longitude' in table_t_id.columns else ibis.null()
        col_ticker = table_t_id['ticker_symbol'] if 'ticker_symbol' in table_t_id.columns else ibis.null()
        col_comp = table_t_id['company_name'] if 'company_name' in table_t_id.columns else ibis.null()
        col_branch = table_t_id['branch_name'] if 'branch_name' in table_t_id.columns else ibis.null()

        table_derived = table_t_id.mutate(
            has_ptaddress = col_pta.notnull(),
            has_ptaddress_latlong = col_lat.notnull() & col_lon.notnull(),
            is_public = col_ticker.notnull(),
            has_company_branch_mismatch = col_comp != col_branch
        ).select(schema_derived_names)

        con.insert("fame_derived", table_derived)
        print(f"✅ Successfully appended derived table from: {ind}")

        for p in ["a2_key_finance", "a3_assets", "a4_profits"]:
            pass

            # TODO: for yearly variables of the form 'my_key 2005', we need to extract the year and somehow store it
            # TODO: then fuzzy match the remainder of the column name to the schema like all the other tables
            # TODO: saving that year for later so we can use it in an unzipped format for the yearly table

    except Exception as e:
        print(f"❌ Error processing folder {ind}")
        print(f"❌ Pipeline failed: {type(e).__name__} - {e}")
        traceback.print_exc() # This prints the full red error log so you know the exact line
    

# Drop all temporary tables
for table_name in con.list_tables():
    # continue if the table name is fame_fixed or fame_derived
    if table_name in ["fame_fixed", "fame_derived", "fame_yearly"]:
        continue
    try:
        con.drop_table(table_name)
        print(f"✅ Successfully dropped table {table_name}.")
    except:
        pass

# Output the head of the fame_fixed and fame_derived tables to verify the data
print("\nHead of fame_derived table:")
print(con.table("fame_derived").execute().head())
print("\nHead of fame_fixed table:")
print(con.table("fame_fixed").execute().head())
print(con.table("fame_fixed").count().execute())

Ingesting industry: 01 with 5 properties.
--- Ingesting property: a1_ID with 5 files.
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
--- Ingesting property: a2_key_finance with 10 files.
--- Ingesting property: a3_assets with 32 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a4_profits with 38 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a5_misc with 3 files.
Deriving ingested files for industry: 01.
⚠️ Mismatch in raw data validation for file 01/a1_ID)
❌ Error processing folder 01
❌ Pipeline failed: IbisTypeError - Arguments registered_number:unknown and registered_number:string are not comparable
Ingesting industry: 02 with 5 properties.


...

C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a4_profits with 4 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a5_misc with 1 files.
Deriving ingested files for industry: 03.
⚠️ Mismatch in raw data validation for file 03/a1_ID)
❌ Error processing folder 03
❌ Pipeline failed: ArrowTypeError - ("Expected bytes, got a 'int' object", 'Conversion failed for column primary_trading_address_no_of_employees with type object')
Ingesting industry: 05 with 5 properties.

Head of fame_derived table:
Empty DataFrame
Columns: [registered_number, has_ptaddress, has_ptaddress_latlong, is_public, has_company_branch_mismatch, industry_code, file_code]
Index: []

Head of fame_fixed table:
Empty DataFrame
Columns: [company_name, registered_number, ticker_symbol, primary_trading_address, primary_trading_address_latitude, primary_trading_address_longitude, branch_name, primary_uk_sic_2007_code, primary_uk_sic_2007_description, latest_accounts_date, no_of_available_years, guo, guo_nb, entity_type]
Index: []
0


Traceback (most recent call last):
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\duckdb\__init__.py", line 1737, in _register_in_memory_table
    obj = data.to_pyarrow_dataset(schema)
          ^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'PandasDataFrameProxy' object has no attribute 'to_pyarrow_dataset'. Did you mean: 'to_pyarrow_bytes'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py", line 124, in <module>
    con.insert("fame_fixed", table_t_fixed_cast)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\sql\__init__.py", line 457, in insert
    self._run_pre_execute_hooks(obj)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\duckdb\__

In [5]:
import traceback

from flask import json
from f_1_traverse import RawFileDict
from f_2_check import drop_duplicate_columns, check_df_matches_schema, handle_excel_dates, rename_df_with_years
from f_2_modify import coerce_ibis_dates_from_schema, reindex_ibis_table
import random

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

# Handle yearly variables
start_year = 2006
end_year = 2025

# We want one big dataframe, which we will merge all the data into for now
# df_fixed = pd.DataFrame(columns=list(schema_fixed.columns))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

process_count = 0
for ind, obj in raw_file_dict.items():
    print(f"Ingesting industry: {ind} with {len(obj)} properties.")
    process_count += 1
    if process_count > 3:
        break

    batches_raw: dict[str, list[pd.DataFrame]] = { p: [] for p in properties }
    for property, arr in obj.items():
        print(f"--- Ingesting property: {property} with {len(arr)} files.")
        schema_raw_fuzzy_col_map = schema_fixed_fuzzy_by_property[property]['col_map']

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for [file_name, file_path] in files_shuffled:

            # LOAD: df_raw has no fixed schema so we can ingest and modify it as pleases
            df_raw = pd.read_excel(file_path, engine='calamine', sheet_name='Results', header=0, dtype={
                "registered_number": str                                            # "Leading Zeros" Trap. Pandas accidentally processes
            })                                                                      #   registered number as int64 when reading the file, which will chop zeros.
            df_raw.drop(df_raw.columns[0], axis=1, inplace=True)                    # Drop column A (blank in raw data)
            df_raw = drop_duplicate_columns(df_raw)                                 # Remove duplicate columns (quirk of some files)

            df_raw = rename_df_with_years(
                df_raw, schema_raw_fuzzy_col_map, property, start_year, end_year,
                ref=f"{ind}/{property}/{file_name}"
            )
            # # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # # Doesn't match the current property or 'all'. This ensures we only keep relevant columns for the current property.
            # p_cols = schema_raw[schema_raw["from_raw"].isin([property, 'all'])]["key"].tolist()
            # df_raw = df_raw[p_cols]

            # Drop columns where the corresponding 'from_raw' entry in the schema_raw
            # Doesn't match the current property or 'all'.
            # Unless the 'keep' column is set to yearly
            # In which case the column name only has to start with the 'key' for that property
            # This ensures we only keep relevant columns for the current property.
            exact_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] != "yearly")]["key"].tolist()
            yearly_cols = schema_raw[(schema_raw["from_raw"].isin([property, 'all'])) & (schema_raw["keep"] == "yearly")]["key"].tolist()
            df_raw_cols = [col for col in df_raw.columns if col in exact_cols or any(col.startswith(yc) for yc in yearly_cols)]
            df_raw = df_raw[df_raw_cols]

            df_raw = df_raw[df_raw['registered_number'].notna()]      
            df_raw = handle_excel_dates(df_raw)                                     # Handle any Excel date serials to datetime
            
            df_raw['industry_code'] = ind
            df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
            batches_raw[property].append(df_raw)

    print(f"Deriving ingested files for industry: {ind}.")

    # START DERIVING
    ind_batches_masters = {
        p: pd.concat(
            batches_raw[p], ignore_index=True
        ).drop_duplicates(                              # Drop duplicates based on 'registered_number'
            subset=['registered_number'], keep='first'
        ).copy()
        for p in properties if batches_raw.get(p)
    }
    try:
        for p in properties:
            if p in ind_batches_masters:
                continue
            raise ValueError(f"❌ Error: No data found for property '{p}' in industry '{ind}'. Please check the raw data files.")

        # FILTER, processing the a1_ID table first.
        schema_raw_fuzzy_mapping = schema_fixed_fuzzy_by_property["a1_ID"]["mapping"]
        df_id = ind_batches_masters["a1_ID"]
        if 'no_of_available_years' in df_id.columns:
            df_id = df_id[df_id['no_of_available_years'] != 0]
        if 'ro_country' in df_id.columns:
            df_id = df_id[df_id['ro_country'] != "Republic of Ireland"]
        ind_batches_masters["a1_ID"] = df_id

        try:
            check_df_matches_schema(schema_raw_fuzzy_mapping, ind_batches_masters["a1_ID"])           # Check if the DataFrame matches the input
        except ValueError as e:
            print(f"⚠️ Mismatch in raw data validation for file {ind}/a1_ID)")
            print("Warning:", e)

        # MODIFY
        table_t_id = ibis.memtable(ind_batches_masters["a1_ID"])
        table_t_misc = ibis.memtable(ind_batches_masters["a5_misc"])
        table_t_merged_id_misc = table_t_id.left_join(table_t_misc, "registered_number").select(
            *[table_t_id[col] for col in table_t_id.columns],
            *[table_t_misc[col] for col in table_t_misc.columns if col not in table_t_id.columns]
        )
        table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_merged_id_misc)
        table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        table_fixed_type_casts = {
            col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
            for col in schema_fixed_names if col in table_t_fixed_dates.columns
        }
        table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        con.insert("fame_fixed", table_t_fixed_cast)
        print(f"✅ Successfully appended fixed table from: {ind}")

        # for p in ["a1_ID", "a5_misc"]: 
        #     df_master = ind_batches_masters[p]
        #     table_t_raw: ibis.expr.types.Table = ibis.memtable(df_master)          # Could be any property, set the raw table into memory

        #     # Filter the memtable to only columns that exist in the target schema
        #     table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_raw)
        #     table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        #     table_fixed_type_casts = {
        #         col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
        #         for col in schema_fixed_names if col in table_t_fixed_dates.columns
        #     }
        #     table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        #     con.insert("fame_fixed", table_t_fixed_cast)
        #     print(f"✅ Successfully appended fixed table from: {ind}")

        # DERIVE
        # for p in ["a1_ID"]:
            # table_derived = table_t_raw.mutate(
            #     has_ptaddress = table_t_raw.primary_trading_address.notnull(),
            #     has_ptaddress_latlong = table_t_raw.primary_trading_address_latitude.notnull()
            #         & table_t_raw.primary_trading_address_longitude.notnull(),
            #     is_public = table_t_raw.ticker_symbol.notnull(),
            #     has_company_branch_mismatch = table_t_raw.company_name != table_t_raw.branch_name
            # ).select(schema_derived_names) # Assuming schema_derived_names is a list of your derived cols
            
            # Safely extract columns, defaulting to an Ibis null object if missing from raw data
        col_pta = table_t_id['primary_trading_address'] if 'primary_trading_address' in table_t_id.columns else ibis.null()
        col_lat = table_t_id['primary_trading_address_latitude'] if 'primary_trading_address_latitude' in table_t_id.columns else ibis.null()
        col_lon = table_t_id['primary_trading_address_longitude'] if 'primary_trading_address_longitude' in table_t_id.columns else ibis.null()
        col_ticker = table_t_id['ticker_symbol'] if 'ticker_symbol' in table_t_id.columns else ibis.null()
        col_comp = table_t_id['company_name'] if 'company_name' in table_t_id.columns else ibis.null()
        col_branch = table_t_id['branch_name'] if 'branch_name' in table_t_id.columns else ibis.null()

        table_derived = table_t_id.mutate(
            has_ptaddress = col_pta.notnull(),
            has_ptaddress_latlong = col_lat.notnull() & col_lon.notnull(),
            is_public = col_ticker.notnull(),
            has_company_branch_mismatch = col_comp != col_branch
        ).select(schema_derived_names)

        con.insert("fame_derived", table_derived)
        print(f"✅ Successfully appended derived table from: {ind}")

        for p in ["a2_key_finance", "a3_assets", "a4_profits"]:
            pass

            # TODO: for yearly variables of the form 'my_key 2005', we need to extract the year and somehow store it
            # TODO: then fuzzy match the remainder of the column name to the schema like all the other tables
            # TODO: saving that year for later so we can use it in an unzipped format for the yearly table

    except Exception as e:
        print(f"❌ Error processing folder {ind}")
        print(f"❌ Pipeline failed: {type(e).__name__} - {e}")
        traceback.print_exc() # This prints the full red error log so you know the exact line
    

# Drop all temporary tables
for table_name in con.list_tables():
    # continue if the table name is fame_fixed or fame_derived
    if table_name in ["fame_fixed", "fame_derived", "fame_yearly"]:
        continue
    try:
        con.drop_table(table_name)
        print(f"✅ Successfully dropped table {table_name}.")
    except:
        pass

# Output the head of the fame_fixed and fame_derived tables to verify the data
print("\nHead of fame_derived table:")
print(con.table("fame_derived").execute().head())
print("\nHead of fame_fixed table:")
print(con.table("fame_fixed").execute().head())
print(con.table("fame_fixed").count().execute())

Ingesting industry: 01 with 5 properties.
--- Ingesting property: a1_ID with 5 files.
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
--- Ingesting property: a2_key_finance with 10 files.
--- Ingesting property: a3_assets with 32 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a4_profits with 38 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a5_misc with 3 files.
Deriving ingested files for industry: 01.
⚠️ Mismatch in raw data validation for file 01/a1_ID)
❌ Error processing folder 01
❌ Pipeline failed: IbisTypeError - Arguments registered_number:unknown and registered_number:string are not comparable
Ingesting industry: 02 with 5 properties.


...

C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a4_profits with 4 files.


C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['industry_code'] = ind
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['file_code'] = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1
C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

--- Ingesting property: a5_misc with 1 files.
Deriving ingested files for industry: 03.
⚠️ Mismatch in raw data validation for file 03/a1_ID)
❌ Error processing folder 03
❌ Pipeline failed: ArrowTypeError - ("Expected bytes, got a 'int' object", 'Conversion failed for column primary_trading_address_no_of_employees with type object')
Ingesting industry: 05 with 5 properties.

Head of fame_derived table:
Empty DataFrame
Columns: [registered_number, has_ptaddress, has_ptaddress_latlong, is_public, has_company_branch_mismatch, industry_code, file_code]
Index: []

Head of fame_fixed table:
Empty DataFrame
Columns: [company_name, registered_number, ticker_symbol, primary_trading_address, primary_trading_address_latitude, primary_trading_address_longitude, branch_name, primary_uk_sic_2007_code, primary_uk_sic_2007_description, latest_accounts_date, no_of_available_years, guo, guo_nb, entity_type]
Index: []
0


Traceback (most recent call last):
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\duckdb\__init__.py", line 1737, in _register_in_memory_table
    obj = data.to_pyarrow_dataset(schema)
          ^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'PandasDataFrameProxy' object has no attribute 'to_pyarrow_dataset'. Did you mean: 'to_pyarrow_bytes'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lazyst\AppData\Local\Temp\ipykernel_30160\1016763081.py", line 124, in <module>
    con.insert("fame_fixed", table_t_fixed_cast)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\sql\__init__.py", line 457, in insert
    self._run_pre_execute_hooks(obj)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\backends\duckdb\__

# 3. [view] resulting DB for inspection

### basic tables overview

In [6]:
# List tables in the DuckDB database as an .md file in /tmp
# Give me the head of all tables
# Ensure they are nicely formatted with headers so I can easily see what's going on
out_file = dirs.root_dir / "build" / "tmp" / "duckdb_tables.md"
with open(out_file, "w") as f:
    tables = con.list_tables()
    f.write("# Tables in DuckDB database\n\n")
    for table in tables:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute()}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).execute().head()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

# Check for any duplicate registered_number values in fame_fixed and fame_derived
for table_name in ["fame_fixed", "fame_derived"]:
    table_df = con.table(table_name).execute()
    duplicate_registered_numbers = table_df[table_df.duplicated(subset=["registered_number"], keep=False)]
    if not duplicate_registered_numbers.empty:
        print(f"⚠️ Warning: Duplicate registered_number values found in {table_name}:")
        print(duplicate_registered_numbers)
    else:
        print(f"No duplicates found in {table_name}.")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duckdb_tables.md
No duplicates found in fame_fixed.
No duplicates found in fame_derived.


In [19]:
# Dump the first 500 rows of df_raw, fame_derived, and famed_fixed to a single .xlsx file in /tmp
# Using those as different sheet names
# Get fame_derived and fame_fixed from their ibis tables
out_file_raw = dirs.root_dir / "build" / "tmp" / "df_raw_head.xlsx"
df_raw_head = df_raw.head(500)
df_derived_head = con.table("fame_derived").execute().head(500)
df_fixed_head = con.table("fame_fixed").execute().head(500)
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    df_derived_head.to_excel(writer, sheet_name='fame_derived', index=False)
    df_fixed_head.to_excel(writer, sheet_name='fame_fixed', index=False)
print(f"✅ Successfully dumped the first 500 rows of df_raw to: {out_file_raw}")

✅ Successfully dumped the first 500 rows of df_raw to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\df_raw_head.xlsx


# 4. Geospatial processing

In [ ]:
import ibis
from f_3_spatial import convert_dms_to_decimal
from f_0_dirs import get_data_dirs

dirs = get_data_dirs()
db_path = dirs.output_dir / "fame_data.duckdb"
# con.raw_sql("INSTALL spatial; LOAD spatial;")

# DERIVE
table_fame_fixed: ibis.expr.types.Table   = con.table("fame_fixed")
table_fame_derived: ibis.expr.types.Table = con.table("fame_derived")
table_joined = table_fame_fixed.left_join(table_fame_derived, "registered_number")

# Assume you load a free UK Postcode to Lat/Lon lookup CSV into DuckDB
# table_postcode_lookup = con.table("uk_postcodes") 

# 2. Mutate Hierarchy & Coords
table_mutated = table_joined.mutate(
    
    # --- ADDRESS HIERARCHY ---
    # Returns the first option that isn't Null
    best_full_address = ibis.coalesce(
        table_fame_fixed.primary_trading_address,
        table_fame_fixed.ro_address,
        # Fallback: concatenate the separate lines if the above are null
        ibis.literal(", ").join(
            ibis.array([
                table_fame_fixed.ro_address_line_1, 
                table_fame_fixed.ro_address_line_2, 
                table_fame_fixed.ro_address_postcode
            ]).filter(lambda x: x.notnull()) # Only join non-null lines
        )
    ),
    
    # --- GEOSPATIAL HIERARCHY ---
    # 1. Parse FAME's DMS strings into pure decimal floats
    fame_lat_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
    fame_lon_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_longitude),
    
    # 2. (Optional Future Step) If you joined a postcode lookup table, 
    # you would include its lat/lon here as a fallback
    # lookup_lat = table_postcode_lookup.latitude,
    
    # 3. Store the best available coordinates
    best_latitude = ibis.coalesce(
        convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
        # lookup_lat
    )
)

# 3. Select Derived Schema Columns and Save
table_derived = table_mutated.select(schema_derived_names)
con.create_table("fame_derived", table_derived, overwrite=True)